# Simple PFN

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tab_utils import PriorDataModule, SimplePFN, SimplePFNClassifier

## Create dataset and loader

In [ ]:
prior = PriorDataModule(
    num_train_batches=4,
    num_val_batches=2,
    num_test_batches=2,
    batch_size=8,
    batch_size_per_gp=4,
    min_features=2,
    max_features=10,
    max_classes=10,
    min_seq_len=None,
    max_seq_len=1024,
    min_train_size=0.1,
    max_train_size=0.9,
    prior_type="mlp_scm",
    num_workers=0,
)

In [ ]:
prior.setup(stage="test")
test_loader = prior.test_dataloader()

batch = next(iter(test_loader))
x = batch[0]  # (batch_size, num_samples, num_features)
y = batch[1]  # (batch_size, num_samples)
num_active_features = batch[2]  # (batch_size,)
num_samples = batch[3]  # (batch_size,)
num_train = batch[4]  # (batch_size,)

print(f"Features shape: {x.shape}")
print(f"Targets shape: {y.shape}")
print(f"Active features: {num_active_features}")
print(f"Total samples: {num_samples}")
print(f"Train samples: {num_train}")

## Run model

In [ ]:
model = SimplePFN(
    num_classes=prior.max_classes,
    num_blocks=1,
    num_heads=2,
    embed_dim=4,
    hidden_dim=6,
    feature_group_size=1,
)

In [ ]:
num_train = SimplePFN._get_value_if_all_equal(num_train)
y_train = y[:, :num_train]
y_test = y[:, num_train:]

y_pred = model(x, y_train)

print(f"Train targets shape: {y_train.shape}")
print(f"Test targets shape: {y_test.shape}")
print(f"Test pred. logits shape: {y_pred.shape}")

## Use sklearn-like interface

In [ ]:
x_train = x[0, :num_train].numpy()
x_test = x[0, num_train:].numpy()
y_train = y[0, :num_train].numpy()
y_test = y[0, num_train:].numpy()

clf = SimplePFNClassifier(model)
clf = clf.fit(x_train, y_train, num_labels=prior.max_classes)
y_probas = clf.predict_proba(x_test)
y_topclass = clf.predict(x_test)

print(f"Train features shape: {x_train.shape}")
print(f"Test features shape: {x_test.shape}")
print(f"Train targets shape: {y_train.shape}")
print(f"Test targets shape: {y_test.shape}")
print(f"Test pred. prob. shape: {y_probas.shape}")
print(f"Test pred. class shape: {y_topclass.shape}")